# Tái hiện benchmark CIRCO của MagicLens

Notebook này chạy lại kết quả **CIRCO** (mAP@5, 10, 25, 50) bằng mã trong repo [google-deepmind/magiclens](https://github.com/google-deepmind/magiclens).

**Điểm khác với FIQ và DTIN.** Nhãn của tập test (800 truy vấn) không được công bố, nên notebook không tự chấm được tập test. Notebook làm hai việc:

1. Chấm thử trên **tập val** (220 truy vấn, có nhãn công khai) ngay trên Colab, để chắc mọi thứ chạy đúng.
2. Tạo tệp top-50 của **tập test** để bạn nộp lên [máy chủ CIRCO](https://circo.micc.unifi.it/) (kết quả gửi qua email) hoặc [máy chủ thay thế trên Codabench](https://www.codabench.org/competitions/10055/).

**Sửa mã để không hết RAM.** Mã gốc giữ cả 123 403 ảnh trong RAM (khoảng 74 GB), gần như chắc chắn làm máy Colab bị dừng. Ô 8 sửa bản sao mã trên máy Colab để đọc ảnh theo từng lô. Hàm tiền xử lý và mô hình giữ nguyên nên kết quả không đổi. Repo của bạn không bị sửa.

## Trước khi chạy

**1. Chọn GPU.** Vào **Runtime → Change runtime type**, chọn **A100 GPU**, bấm **Save**.

**2. Không cần chuẩn bị dữ liệu.** Notebook tự tải nhãn CIRCO và khoảng 19 GB ảnh COCO ở mỗi phiên. Máy Colab cần khoảng 45 GB ổ đĩa trống để tải và giải nén; ô 3 sẽ in dung lượng còn trống.

## Cách chạy

- Chạy lần lượt từng ô từ trên xuống: bấm vào ô rồi nhấn **Shift + Enter**, hoặc bấm nút ▶ bên trái ô. Muốn chạy hết một lượt thì chọn **Runtime → Run all**.
- Ô 4 sẽ hỏi quyền truy cập Google Drive: chọn tài khoản của bạn và đồng ý.
- Kết quả (bảng so sánh, log, tệp top-50) được lưu vào `MyDrive/magiclens/ket_qua/circo/` trên Google Drive.
- Mỗi lần mở lại notebook là một máy Colab mới, nên phải chạy lại từ ô 1. Trong cùng một phiên, ô nào đã làm xong sẽ tự bỏ qua.
- Pip có thể in cảnh báo bắt đầu bằng `ERROR: pip's dependency resolver...`. Đây chỉ là cảnh báo xung đột phiên bản với các gói có sẵn của Colab, **bỏ qua được**. Notebook chỉ dừng khi có lỗi thật.
- Xong việc thì vào **Runtime → Disconnect and delete runtime** để không tốn compute unit.

## Bước 1: Cấu hình

In [ ]:
# Ô 1: Cấu hình. Chỉ cần sửa ở ô này.
MODEL_SIZE = "base"          # "base" hoặc "large"
BATCH_SIZE = None            # None = tự chọn; giảm (ví dụ 256) nếu báo RESOURCE_EXHAUSTED
CHAY_VAL = True              # chấm thử trên tập val (220 truy vấn, có nhãn) ngay trên Colab
CHAY_TEST = True             # tạo tệp top-50 của tập test (800 truy vấn) để nộp lên máy chủ CIRCO

DRIVE_DIR = "/content/drive/MyDrive/magiclens"   # thư mục làm việc trên Google Drive
REPO = "/content/magiclens"                      # nơi đặt mã nguồn trên máy Colab
BENCHMARK = "circo"

## Bước 2: Chuẩn bị máy Colab

Các ô 2–7 kiểm tra máy, kết nối Drive, cài thư viện, tải checkpoint và chạy thử mô hình. Ở ô 7, dòng cuối phải có `chạy trên: GPU`. Nếu là `CPU` thì runtime chưa có GPU hoặc JAX chưa nhận GPU: kiểm tra lại ô 3 và ô 5.

In [ ]:
# Ô 2: Các hàm dùng chung. Chạy ô này, không cần sửa gì.
import ast, codecs, glob, json, os, re, shlex, shutil, subprocess, sys, tarfile, time, zipfile

try:
    import google.colab  # có mặt khi chạy trên Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"           # bớt log của TensorFlow
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"   # TensorFlow không giữ trước bộ nhớ GPU
PY = sys.executable                                 # trình Python dùng để cài thư viện và chạy mã
CONTENT = os.path.dirname(REPO)                     # /content trên Colab
LOG_DIR = os.path.join(CONTENT, "logs")
os.makedirs(LOG_DIR, exist_ok=True)
q = shlex.quote


def sh(cmd, log=None, check=True):
    # Chạy một lệnh shell, in kết quả ngay khi có và ghi vào tệp log nếu cần.
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    dec = codecs.getincrementaldecoder("utf-8")(errors="replace")
    f = open(log, "wb") if log else None
    try:
        while True:
            chunk = p.stdout.read1(65536)
            if not chunk:
                break
            sys.stdout.write(dec.decode(chunk))
            sys.stdout.flush()
            if f:
                f.write(chunk)
    finally:
        if f:
            f.close()
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"Lệnh bị lỗi (mã {rc}). Đọc thông báo ngay phía trên để biết nguyên nhân.")
    return rc


def chay_python(ma, cwd=None):
    # Ghi đoạn mã ra tệp rồi chạy bằng một tiến trình Python riêng.
    thu_muc = cwd or CONTENT
    tep = os.path.join(thu_muc, "_chay_thu.py")
    with open(tep, "w", encoding="utf-8") as f:
        f.write(ma)
    return sh(f"cd {q(thu_muc)} && {q(PY)} {q(tep)}")


def ket_qua_dir():
    d = os.path.join(DRIVE_DIR, "ket_qua", BENCHMARK, MODEL_SIZE)
    os.makedirs(d, exist_ok=True)
    return d


def chay_inference(dataset, ten_log, env="", output="./output"):
    # Chạy inference.py của repo cho một bộ dữ liệu, ghi log, trả về đường dẫn log.
    bs = BATCH_SIZE or (1024 if MODEL_SIZE == "base" else 512)
    log = os.path.join(LOG_DIR, ten_log)
    cmd = (f"cd {q(REPO)} && {env} {q(PY)} inference.py --model_size {MODEL_SIZE} "
           f"--model_path ./models/magic_lens_clip_{MODEL_SIZE}.pkl --dataset {dataset} "
           f"--batch_size {bs} --output {output}")
    t0 = time.time()
    sh(cmd, log=log)
    print(f"\nXong {dataset} sau {(time.time() - t0) / 60:.1f} phút. Log: {log}")
    shutil.copy(log, ket_qua_dir())
    return log


def doc_recall(log):
    # Đọc dòng "Recalls:  {10: ..., 50: ...}" mà inference.py in ra.
    text = open(log, encoding="utf-8", errors="replace").read()
    found = re.findall(r"Recalls:\s*(\{[^}]*\})", text)
    if not found:
        raise RuntimeError(f"Không thấy dòng 'Recalls:' trong {log}.")
    r = ast.literal_eval(found[-1])
    return r[10], r[50]


def bang(tieu_de, cac_dong):
    # Tạo bảng Markdown.
    dong = ["| " + " | ".join(tieu_de) + " |", "|" + "---|" * len(tieu_de)]
    dong += ["| " + " | ".join(str(x) for x in d) + " |" for d in cac_dong]
    return "\n".join(dong)


def hien_thi(md):
    try:
        from IPython.display import Markdown, display
        display(Markdown(md))
    except ImportError:
        print(md)


def con_trong_gb(path):
    return shutil.disk_usage(path).free / 1e9


print("Sẵn sàng. Đang chạy trên Colab:", IN_COLAB)

In [ ]:
# Ô 3: Kiểm tra máy Colab: GPU, RAM, ổ đĩa
if sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", check=False) != 0:
    print("\nCẢNH BÁO: không thấy GPU. Vào Runtime → Change runtime type → chọn A100 GPU → Save, rồi chạy lại từ ô 1.")
sh("free -g | head -2", check=False)
print(f"Ổ đĩa còn trống: {con_trong_gb(CONTENT):.0f} GB")
print("CIRCO cần khoảng 45 GB ổ đĩa trống để tải và giải nén 19 GB ảnh COCO.")

In [ ]:
# Ô 4: Kết nối Google Drive. Colab sẽ hỏi quyền truy cập: chọn tài khoản của bạn và đồng ý.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Thư mục làm việc trên Drive:", DRIVE_DIR)

In [ ]:
# Ô 5: Tải mã nguồn và cài thư viện. Lần đầu mất vài phút; chạy lại trong cùng phiên sẽ tự bỏ qua.
DA_CAI = os.path.join(CONTENT, ".magiclens_da_cai")
if os.path.exists(DA_CAI):
    print("Đã cài trong phiên này, bỏ qua.")
else:
    if not os.path.isdir(REPO):
        sh(f"git clone -q https://github.com/google-deepmind/magiclens.git {q(REPO)}")
    scenic = os.path.join(CONTENT, "scenic")
    if not os.path.isdir(scenic):
        sh(f"git clone -q https://github.com/google-research/scenic.git {q(scenic)}")
    pip = f"{q(PY)} -m pip install -q"
    sh(f"{pip} {q(scenic)}")
    sh(f"{pip} -r {q(scenic + '/scenic/projects/baselines/clip/requirements.txt')}")
    # Bộ phiên bản đã chạy được với mã này. Nếu không cài được thì dùng bản mới nhất.
    if sh(f'{pip} "jax[cuda12]==0.11.1" "flax==0.12.9"', check=False) != 0:
        print("Không cài được bộ phiên bản đã kiểm thử, chuyển sang bản mới nhất.")
        sh(f'{pip} -U "jax[cuda12]" flax')
    open(DA_CAI, "w").close()
chay_python("import jax, flax\nprint('jax', jax.__version__, '| flax', flax.__version__, '|', jax.devices())\n")

In [ ]:
# Ô 6: Tải checkpoint MagicLens (công khai, không cần đăng nhập). Bản base 666 MB, bản large 1.9 GB.
KICH_THUOC = {"base": 665747853, "large": 1861692696}
CKPT = os.path.join(REPO, "models", f"magic_lens_clip_{MODEL_SIZE}.pkl")
if os.path.exists(CKPT) and os.path.getsize(CKPT) == KICH_THUOC[MODEL_SIZE]:
    print("Đã có checkpoint:", CKPT)
else:
    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    url = f"https://storage.googleapis.com/gresearch/magiclens/models/magic_lens_clip_{MODEL_SIZE}.pkl"
    sh(f"wget -q --show-progress -c -O {q(CKPT)} {url}")
    if os.path.getsize(CKPT) != KICH_THUOC[MODEL_SIZE]:
        raise RuntimeError("Tệp tải về chưa đủ dung lượng. Chạy lại ô này để tải tiếp.")
    print("Đã tải:", CKPT)

In [ ]:
# Ô 7: Chạy thử mô hình một lần để chắc mọi thứ đã sẵn sàng
chay_python(
    "import jax, jax.numpy as jnp\n"
    "from inference import load_model\n"
    f"model, params = load_model('{MODEL_SIZE}', 'models/magic_lens_clip_{MODEL_SIZE}.pkl')\n"
    "out = model.apply(params, {'ids': jnp.ones((1, 1, 77), jnp.int32), 'image': jnp.ones((1, 224, 224, 3))})\n"
    "print('Embedding:', out['multimodal_embed_norm'].shape, '| chạy trên:', jax.devices()[0].platform.upper())\n",
    cwd=REPO)

## Bước 3: Sửa mã và tải dữ liệu

Ô 8 sửa hai tệp trong bản sao mã trên máy Colab:

- `data_utils.py`: kho ảnh CIRCO chỉ lưu đường dẫn thay vì ảnh, và tập nhãn được chọn qua biến môi trường `CIRCO_SPLIT` (mặc định `test`, như mã gốc).
- `inference.py`: khi mã hóa kho ảnh, đọc ảnh theo từng lô bằng đúng hàm `process_img` của mã gốc.

Ô 9 tải nhãn CIRCO và ảnh COCO unlabeled 2017.

In [ ]:
# Ô 8: Sửa mã để đọc ảnh CIRCO theo lô (chỉ sửa bản sao trên máy Colab)
DAU = "# [notebook] doc anh CIRCO theo lo"
SUA = {
    "data_utils.py": [
        ("        img_path = os.path.join(index_image_folder, image_id2name(index_img_id))\n"
         "        ima = process_img(img_path, 224)\n"
         "        return IndexExample(iid=index_img_id, iimage=ima, itokens=null_tokens)\n",
         "        img_path = os.path.join(index_image_folder, image_id2name(index_img_id))\n"
         "        return IndexExample(iid=index_img_id, iimage=img_path, itokens=null_tokens)  " + DAU + "\n"),
        ('queries = json.load(open("./data/circo/annotations/test.json"))',
         'queries = json.load(open(f"./data/circo/annotations/{os.environ.get(\'CIRCO_SPLIT\', \'test\')}.json"))'),
    ],
    "inference.py": [
        ("from data_utils import build_circo_dataset, build_fiq_dataset, build_dtin_dataset\n",
         "from data_utils import build_circo_dataset, build_fiq_dataset, build_dtin_dataset\n"
         "from data_utils import process_img  " + DAU + "\n"
         "from concurrent.futures import ThreadPoolExecutor\n"
         "_POOL = ThreadPoolExecutor()\n"),
        ("    iimages = jnp.concatenate([i.iimage for i in batch], axis=0)\n",
         "    iimages = [x.iimage for x in batch]\n"
         "    if isinstance(iimages[0], str):  # CIRCO: kho anh chi luu duong dan, doc anh theo lo\n"
         "      iimages = list(_POOL.map(lambda p: process_img(p, 224), iimages))\n"
         "    iimages = jnp.concatenate(iimages, axis=0)\n"),
    ],
}
for ten, cac_cho in SUA.items():
    duong_dan = os.path.join(REPO, ten)
    s = open(duong_dan, encoding="utf-8").read()
    if DAU in s:
        print(f"{ten}: đã sửa từ trước, bỏ qua.")
        continue
    for cu, moi in cac_cho:
        if s.count(cu) != 1:
            raise RuntimeError(f"Không tìm thấy đoạn mã cần sửa trong {ten}. Mã gốc có thể đã thay đổi.")
        s = s.replace(cu, moi)
    with open(duong_dan, "w", encoding="utf-8") as f:
        f.write(s)
    print(f"{ten}: đã sửa.")

In [ ]:
# Ô 9: Tải dữ liệu CIRCO. Ảnh COCO khoảng 19 GB, chỉ tải khi máy Colab chưa có.
CIRCO_DIR = os.path.join(REPO, "data", "circo")
COCO_DIR = os.path.join(CIRCO_DIR, "COCO2017_unlabeled")
ANH_DIR = os.path.join(COCO_DIR, "unlabeled2017")
INFO = os.path.join(COCO_DIR, "annotations", "image_info_unlabeled2017.json")
os.makedirs(COCO_DIR, exist_ok=True)

if not os.path.exists(os.path.join(CIRCO_DIR, "annotations", "test.json")):
    tam = os.path.join(CONTENT, "CIRCO_repo")
    if not os.path.isdir(tam):
        sh(f"git clone -q https://github.com/miccunifi/CIRCO.git {q(tam)}")
    shutil.copytree(os.path.join(tam, "annotations"), os.path.join(CIRCO_DIR, "annotations"), dirs_exist_ok=True)
    shutil.rmtree(tam)

if not os.path.exists(INFO):
    z = os.path.join(CONTENT, "image_info_unlabeled2017.zip")
    sh(f"wget -q -c -O {q(z)} http://images.cocodataset.org/annotations/image_info_unlabeled2017.zip")
    sh(f"unzip -q -o {q(z)} -d {q(COCO_DIR)}")
    os.remove(z)

SO_ANH = len(json.load(open(INFO))["images"])
co = len(os.listdir(ANH_DIR)) if os.path.isdir(ANH_DIR) else 0
if co < SO_ANH:
    if con_trong_gb(CONTENT) < 45:
        raise RuntimeError(f"Ổ đĩa chỉ còn {con_trong_gb(CONTENT):.0f} GB, cần khoảng 45 GB để tải và giải nén ảnh COCO.")
    z = os.path.join(CONTENT, "unlabeled2017.zip")
    sh(f"wget -q --show-progress -c -O {q(z)} http://images.cocodataset.org/zips/unlabeled2017.zip")
    sh(f"unzip -q -o {q(z)} -d {q(COCO_DIR)}")
    os.remove(z)
    co = len(os.listdir(ANH_DIR))
print(f"Ảnh COCO: {co}/{SO_ANH} (bản đầy đủ: 123403)")
if co < SO_ANH:
    raise RuntimeError("Thiếu ảnh COCO. Chạy lại ô này.")

## Bước 4: Chạy đánh giá

Ô 10 chấm thử trên tập val. Ô 11 tạo tệp kết quả tập test để nộp. Mỗi ô mã hóa lại toàn bộ 123 403 ảnh, nên đây là phần chạy lâu nhất. Nếu chỉ cần tệp để nộp, đặt `CHAY_VAL = False` ở ô 1.

In [ ]:
# Ô 10: Chấm thử trên tập val (220 truy vấn, nhãn công khai)
def map_at_k(tep_ket_qua, tep_nhan):
    # mAP@K theo định nghĩa của CIRCO: AP@K = tổng P@k * rel(k) với k <= K, chia cho min(K, số ảnh đúng).
    nhan = {str(x["id"]): set(x["gt_img_ids"]) for x in json.load(open(tep_nhan))}
    ket_qua = json.load(open(tep_ket_qua))
    out = {}
    for K in (5, 10, 25, 50):
        aps = []
        for qid, dung in nhan.items():
            trung, tong = 0, 0.0
            for k, anh in enumerate(ket_qua[qid][:K], 1):
                if anh in dung:
                    trung += 1
                    tong += trung / k
            aps.append(tong / min(K, len(dung)))
        out[K] = 100 * sum(aps) / len(aps)
    return out


MAP_VAL = None
if CHAY_VAL:
    chay_inference("circo", f"circo-val_{MODEL_SIZE}.log", env="CIRCO_SPLIT=val", output="./output_val")
    tep_val = os.path.join(REPO, "output_val", f"circo_{MODEL_SIZE}", "circo_results.json")
    MAP_VAL = map_at_k(tep_val, os.path.join(CIRCO_DIR, "annotations", "val.json"))
    shutil.copy(tep_val, os.path.join(ket_qua_dir(), f"circo_val_{MODEL_SIZE}.json"))
    hien_thi(bang(["Tập val (220 truy vấn)", "mAP@5", "mAP@10", "mAP@25", "mAP@50"],
                  [[f"MagicLens-{MODEL_SIZE}"] + [f"{MAP_VAL[k]:.1f}" for k in (5, 10, 25, 50)]]))
    print("Bài báo không báo cáo tập val. Điểm phải cùng cỡ với số của tập test ở ô 12, không gần 0.")
else:
    print("Bỏ qua tập val (CHAY_VAL = False).")

In [ ]:
# Ô 11: Tạo tệp top-50 của tập test để nộp lên máy chủ CIRCO
TEP_NOP = None
if CHAY_TEST:
    chay_inference("circo", f"circo-test_{MODEL_SIZE}.log")
    tep_test = os.path.join(REPO, "output", f"circo_{MODEL_SIZE}", "circo_results.json")
    ket_qua = json.load(open(tep_test))
    so_truy_van = len(json.load(open(os.path.join(CIRCO_DIR, "annotations", "test.json"))))
    if len(ket_qua) != so_truy_van or any(len(v) != min(50, SO_ANH) for v in ket_qua.values()):
        raise RuntimeError("Tệp kết quả không đủ truy vấn hoặc không đủ 50 ảnh cho mỗi truy vấn.")
    TEP_NOP = os.path.join(ket_qua_dir(), f"circo_test_{MODEL_SIZE}.json")
    shutil.copy(tep_test, TEP_NOP)
    with zipfile.ZipFile(TEP_NOP[:-5] + ".zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(TEP_NOP, arcname=os.path.basename(TEP_NOP))
    print(f"Tệp để nộp: {TEP_NOP} ({so_truy_van} truy vấn × top-50)")
    print("Bản nén cho Codabench:", TEP_NOP[:-5] + ".zip")
else:
    print("Bỏ qua tập test (CHAY_TEST = False).")

## Bước 5: Nộp kết quả và so sánh

Tệp để nộp nằm trong `MyDrive/magiclens/ket_qua/circo/<base|large>/` trên Drive:

1. Tải tệp `circo_test_<base|large>.json` từ Drive về máy.
2. Nộp lên [máy chủ CIRCO](https://circo.micc.unifi.it/); kết quả được gửi về qua email. Hoặc nộp tệp `.zip` cùng tên lên [Codabench](https://www.codabench.org/competitions/10055/).
3. So với bảng do ô 12 in ra. Nên so với dòng "checkpoint chuyển đổi", vì đó là checkpoint bạn đang dùng.

In [ ]:
# Ô 12: Số tham chiếu của tập test và lưu tổng hợp lên Drive
THAM_CHIEU = {
    "base":  [("Bài báo (Bảng 14)", (23.1, 23.8, 25.8, 26.7)), ("Checkpoint chuyển đổi (README)", (22.3, 23.2, 25.0, 26.0))],
    "large": [("Bài báo (Bảng 14)", (29.6, 30.8, 33.4, 34.4)), ("Checkpoint chuyển đổi (README)", (29.5, 30.8, 33.2, 34.3))],
}[MODEL_SIZE]
cac_dong = [[f"Tập test: {ten}"] + list(so) for ten, so in THAM_CHIEU]
cac_dong.append(["Tập test: tái hiện", "(điền sau khi máy chủ trả kết quả)", "", "", ""])
if MAP_VAL:
    cac_dong.append(["Tập val: tái hiện"] + [f"{MAP_VAL[k]:.1f}" for k in (5, 10, 25, 50)])
md_kq = f"### CIRCO, MagicLens-{MODEL_SIZE} (mAP)\n\n" + bang(["", "mAP@5", "mAP@10", "mAP@25", "mAP@50"], cac_dong)
hien_thi(md_kq)
with open(os.path.join(ket_qua_dir(), "tong_hop.md"), "w", encoding="utf-8") as f:
    f.write(md_kq + "\n")
print("Đã lưu vào", ket_qua_dir())

## Chạy bản Large

Ở ô 1, đổi `MODEL_SIZE = "large"`, chạy lại ô 1, rồi chạy tiếp từ ô 6 trở xuống. Dữ liệu đã chuẩn bị trong phiên này được dùng lại. Với CIRCO, ô 8 và ảnh COCO không cần làm lại.

## Kết thúc

Kết quả đã nằm trên Drive. Để không tốn compute unit, vào **Runtime → Disconnect and delete runtime**, hoặc đổi `NGAT_KET_NOI = True` ở ô dưới rồi chạy ô đó.

In [ ]:
# Ô cuối: Ngắt kết nối máy Colab. Mọi thứ trên máy sẽ bị xóa, kết quả trên Drive vẫn còn.
NGAT_KET_NOI = False   # đổi thành True rồi chạy ô này khi đã xong
if NGAT_KET_NOI and IN_COLAB:
    from google.colab import runtime
    runtime.unassign()